# Chapter 13 - Boosting

While bagging builds trees **independently** and averages them, boosting builds trees
**sequentially** — each new tree focuses on the mistakes of the previous ones. This
sequential correction strategy often produces the most accurate models on tabular data,
which is why gradient boosting dominates machine learning competitions.

**Topics covered:**
- Boosting concept: sequential learning from mistakes
- AdaBoost: adjusting sample weights based on errors
- AdaBoostClassifier in sklearn
- Gradient Boosting: fitting residuals sequentially
- GradientBoostingClassifier and GradientBoostingRegressor
- Key hyperparameters: n_estimators, learning_rate, max_depth, subsample
- Learning rate and n_estimators tradeoff (staged predictions)
- HistGradientBoosting (sklearn's fast implementation)
- Brief mention of XGBoost, LightGBM, CatBoost
- Comparing: Decision Tree vs Random Forest vs Gradient Boosting
- Model comparison visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_squared_error,
    ConfusionMatrixDisplay,
)

np.random.seed(42)
sns.set_style('whitegrid')
%matplotlib inline

---
## 1. The Boosting Idea

Boosting is based on a simple but powerful insight: instead of building many independent
models (as in bagging), build models **sequentially** so that each new model corrects
the errors of the previous ensemble.

The two main flavours are:

1. **AdaBoost** (Adaptive Boosting): adjusts sample **weights** so that misclassified
   samples get more attention from the next tree.
2. **Gradient Boosting**: fits each new tree to the **residuals** (negative gradient of
   the loss) left by the previous trees.

Both start with a weak learner (typically a shallow tree, called a *stump* for depth 1)
and gradually build up a strong learner.

In [ ]:
# Prepare data for demonstrations
X_moons, y_moons = make_moons(n_samples=500, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=42
)

# Mesh for decision boundaries
x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

---
## 2. AdaBoost: Adaptive Boosting

AdaBoost (Freund & Schapire, 1997) works as follows:

1. Assign equal weights to all training samples: $w_i = 1/n$.
2. Train a weak learner (e.g., a decision stump) on the weighted data.
3. Compute the weighted error rate $\epsilon$.
4. Compute the learner's weight: $\alpha = \frac{1}{2} \ln\frac{1 - \epsilon}{\epsilon}$.
   Accurate learners get higher $\alpha$.
5. Update sample weights: increase weights for misclassified samples, decrease for
   correctly classified ones.
6. Repeat for the next learner.
7. Final prediction: weighted vote of all learners.

The key insight is that each successive learner focuses on the "hard" examples that
previous learners got wrong.

In [ ]:
# Visualise how AdaBoost reweights samples over iterations
# We will manually track the sample weights
ada_demo = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=3,
    learning_rate=1.0,
    algorithm='SAMME',
    random_state=42
)

# Fit step by step to show weight evolution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sample_weight = np.ones(len(X_train)) / len(X_train)

for idx in range(3):
    # Train one stump on current weights
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    stump.fit(X_train, y_train, sample_weight=sample_weight)
    y_pred = stump.predict(X_train)

    # Compute weighted error
    incorrect = (y_pred != y_train)
    err = np.sum(sample_weight * incorrect) / np.sum(sample_weight)
    alpha = 0.5 * np.log((1 - err) / max(err, 1e-10))

    # Visualise
    ax = axes[idx]
    Z = stump.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
    # Size proportional to weight
    sizes = 2000 * sample_weight / sample_weight.max()
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu',
               edgecolors='k', s=sizes, alpha=0.7)
    ax.set_title(f'Stump {idx + 1} | error={err:.3f} | alpha={alpha:.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

    # Update weights
    sample_weight *= np.exp(alpha * np.where(incorrect, 1, -1))
    sample_weight /= sample_weight.sum()

plt.suptitle('AdaBoost: sample weights grow for misclassified points (larger dots = higher weight)',
             fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

Notice how the dot sizes change: after each stump, the misclassified samples become
larger (higher weight), forcing the next stump to focus on them.

---
## 3. AdaBoostClassifier in Scikit-Learn

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_train, y_train)

print(f"AdaBoost — Train: {ada.score(X_train, y_train):.4f}  "
      f"Test: {ada.score(X_test, y_test):.4f}")

In [ ]:
# Show AdaBoost decision boundary at different numbers of estimators
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, n_est in zip(axes, [1, 10, 50, 200]):
    ada_temp = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n_est,
        learning_rate=0.5,
        random_state=42
    )
    ada_temp.fit(X_train, y_train)
    Z = ada_temp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu',
               edgecolors='k', s=30)
    ax.set_title(f'n_estimators={n_est}\nTest: {ada_temp.score(X_test, y_test):.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('AdaBoost decision boundary at increasing ensemble size', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Even though each individual stump can only make a single straight-line cut, combining
hundreds of them produces a complex, smooth decision boundary.

---
## 4. Gradient Boosting: Fitting Residuals

Gradient boosting (Friedman, 2001) takes a different approach from AdaBoost. Instead
of reweighting samples, it fits each new tree to the **residuals** (more precisely, the
negative gradient of the loss function) left by the current ensemble.

The algorithm for regression with squared error loss:

1. Start with an initial prediction $F_0(x) = \bar{y}$ (the mean).
2. For $m = 1, 2, \ldots, M$:
   - Compute residuals: $r_i = y_i - F_{m-1}(x_i)$
   - Fit a tree $h_m$ to the residuals.
   - Update: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$
3. Final prediction: $F_M(x)$.

The **learning rate** $\eta$ (typically 0.01 to 0.3) scales each tree's contribution.
Smaller learning rates require more trees but generally produce better results due to
stronger regularisation.

In [ ]:
# Demonstrate gradient boosting step by step on a regression problem
np.random.seed(42)
X_gb = np.sort(5 * np.random.rand(100, 1), axis=0)
y_gb = np.sin(X_gb).ravel() + 0.3 * np.random.randn(100)

X_plot_gb = np.linspace(0, 5, 300).reshape(-1, 1)
y_true = np.sin(X_plot_gb).ravel()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Manual gradient boosting steps
learning_rate = 0.5
F = np.full(len(y_gb), np.mean(y_gb))  # initial prediction
F_plot = np.full(len(X_plot_gb), np.mean(y_gb))

for i in range(6):
    ax = axes.ravel()[i]

    # Compute residuals
    residuals = y_gb - F

    # Fit tree to residuals
    tree = DecisionTreeRegressor(max_depth=2, random_state=42)
    tree.fit(X_gb, residuals)

    # Update predictions
    F += learning_rate * tree.predict(X_gb)
    F_plot += learning_rate * tree.predict(X_plot_gb)

    # Plot
    ax.scatter(X_gb, y_gb, s=20, alpha=0.5, label='Data')
    ax.plot(X_plot_gb, F_plot, color='red', linewidth=2, label='Ensemble')
    ax.plot(X_plot_gb, y_true, color='green', linewidth=1.5,
            linestyle='--', label='True')
    mse = np.mean((y_gb - F) ** 2)
    ax.set_title(f'After tree {i + 1} | MSE: {mse:.4f}')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$y$')
    if i == 0:
        ax.legend(loc='best', fontsize=8)

plt.suptitle('Gradient boosting: each tree corrects the residuals of the previous ensemble',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

With each tree, the ensemble gets closer to the true function. The residuals shrink,
and the MSE decreases. This is the core mechanism of gradient boosting.

---
## 5. GradientBoostingClassifier and GradientBoostingRegressor

In [ ]:
# Classification
gb_clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_clf.fit(X_train, y_train)

print(f"Gradient Boosting — Train: {gb_clf.score(X_train, y_train):.4f}  "
      f"Test: {gb_clf.score(X_test, y_test):.4f}")

In [ ]:
# Regression
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(300, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + 0.3 * np.random.randn(300)

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

gb_reg = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_reg.fit(X_tr_r, y_tr_r)

X_plot_r = np.linspace(0, 5, 500).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(X_tr_r, y_tr_r, s=15, alpha=0.4, label='Train')
ax.plot(X_plot_r, gb_reg.predict(X_plot_r), color='red', linewidth=2,
        label='Gradient Boosting')
ax.plot(X_plot_r, np.sin(X_plot_r), color='green', linewidth=1.5,
        linestyle='--', label='True function')
mse = mean_squared_error(y_te_r, gb_reg.predict(X_te_r))
ax.set_title(f'GradientBoostingRegressor | Test MSE: {mse:.4f}')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Key Hyperparameters

| Parameter | Description | Typical values |
|-----------|-------------|----------------|
| `n_estimators` | Number of boosting stages (trees) | 100 - 5000 |
| `learning_rate` | Shrinkage factor for each tree | 0.01 - 0.3 |
| `max_depth` | Maximum depth of each tree | 3 - 8 (shallow trees!) |
| `subsample` | Fraction of samples used per tree | 0.5 - 1.0 |
| `min_samples_leaf` | Minimum samples in each leaf | 1 - 20 |

Unlike random forests (which use deep trees), gradient boosting typically uses **shallow
trees** (depth 3-5). The ensemble's power comes from the sequential correction, not
from individual tree complexity.

In [ ]:
# Effect of max_depth on gradient boosting
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, depth in zip(axes, [1, 3, 5, 10]):
    gb = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=depth, random_state=42
    )
    gb.fit(X_train, y_train)
    Z = gb.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu',
               edgecolors='k', s=30)
    ax.set_title(f'max_depth={depth}\nTest: {gb.score(X_test, y_test):.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Gradient boosting: effect of max_depth', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Learning Rate and n_estimators Tradeoff

The learning rate and number of trees are closely linked:
- A **small learning rate** (e.g., 0.01) makes each tree contribute very little, so you
  need many more trees to reach the same performance.
- A **large learning rate** (e.g., 1.0) makes each tree contribute a lot, but the model
  can overshoot and overfit.

The rule of thumb: **lower the learning rate and increase n_estimators**. The sweet spot
trades off compute budget against model quality.

We can use `staged_predict` to see how the ensemble evolves as trees are added.

In [ ]:
# Learning rate vs n_estimators tradeoff
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

learning_rates = [0.01, 0.05, 0.1, 0.3, 1.0]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(learning_rates)))

for lr, color in zip(learning_rates, colors):
    gb = GradientBoostingClassifier(
        n_estimators=500, learning_rate=lr, max_depth=3, random_state=42
    )
    gb.fit(X_train, y_train)

    # Staged predictions on train and test
    train_staged = [accuracy_score(y_train, pred)
                    for pred in gb.staged_predict(X_train)]
    test_staged = [accuracy_score(y_test, pred)
                   for pred in gb.staged_predict(X_test)]

    axes[0].plot(range(1, 501), train_staged, color=color, alpha=0.7,
                label=f'lr={lr}')
    axes[1].plot(range(1, 501), test_staged, color=color, alpha=0.7,
                label=f'lr={lr}')

axes[0].set_title('Training accuracy')
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].set_title('Test accuracy')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('Learning rate vs n_estimators: staged predictions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Key observations:
- **lr=1.0** reaches peak test accuracy quickly but may overfit with more trees.
- **lr=0.01** needs many more trees to converge but can achieve excellent performance.
- For the test set, the smaller learning rates are more stable and less prone to
  overfitting with more trees.

In [ ]:
# Staged prediction for regression (MSE)
gb_reg_staged = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.1, max_depth=3, random_state=42
)
gb_reg_staged.fit(X_tr_r, y_tr_r)

train_mse = [mean_squared_error(y_tr_r, pred)
             for pred in gb_reg_staged.staged_predict(X_tr_r)]
test_mse = [mean_squared_error(y_te_r, pred)
            for pred in gb_reg_staged.staged_predict(X_te_r)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, 501), train_mse, label='Train MSE', linewidth=2)
ax.plot(range(1, 501), test_mse, label='Test MSE', linewidth=2)
best_n = np.argmin(test_mse) + 1
ax.axvline(best_n, color='red', linestyle='--',
           label=f'Best n_estimators = {best_n}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('MSE')
ax.set_title('Staged MSE: train vs test (GradientBoostingRegressor)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Optimal n_estimators: {best_n}")
print(f"Minimum test MSE:     {min(test_mse):.6f}")

The test MSE decreases initially and then starts increasing — classic overfitting. In
practice, you would use early stopping or cross-validation to find the optimal number
of trees.

---
## 8. Subsample: Stochastic Gradient Boosting

Setting `subsample < 1.0` trains each tree on a random fraction of the training data
(without replacement). This introduces randomness similar to bagging, which can improve
generalisation and also speeds up training.

In [ ]:
# Effect of subsample
subsample_values = [0.5, 0.7, 0.9, 1.0]

fig, ax = plt.subplots(figsize=(9, 5))

for ss in subsample_values:
    gb = GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=3,
        subsample=ss, random_state=42
    )
    gb.fit(X_train, y_train)
    test_staged = [accuracy_score(y_test, pred)
                   for pred in gb.staged_predict(X_test)]
    ax.plot(range(1, 301), test_staged, label=f'subsample={ss}', linewidth=1.5)

ax.set_xlabel('n_estimators')
ax.set_ylabel('Test accuracy')
ax.set_title('Effect of subsample on test accuracy')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. HistGradientBoosting: Sklearn's Fast Implementation

Scikit-learn's `HistGradientBoostingClassifier` and `HistGradientBoostingRegressor` are
inspired by LightGBM. They use **histogram-based splitting**, which bins continuous
features into discrete values (up to 255 bins). This provides:

- **Much faster training** on large datasets (orders of magnitude for n > 10,000)
- **Native support for missing values**
- **Categorical feature support** (via `categorical_features` parameter)
- **Built-in early stopping** via `early_stopping='auto'`

For datasets larger than a few thousand samples, `HistGradientBoosting*` is almost
always preferred over `GradientBoosting*`.

In [ ]:
# Compare training time: GradientBoosting vs HistGradientBoosting
import time

# Larger dataset for timing comparison
X_large, y_large = make_classification(
    n_samples=10000, n_features=20, n_informative=10,
    random_state=42
)
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_large, y_large, test_size=0.25, random_state=42
)

# Standard Gradient Boosting
start = time.time()
gb_standard = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42
)
gb_standard.fit(X_tr_l, y_tr_l)
time_standard = time.time() - start

# HistGradientBoosting
start = time.time()
hgb = HistGradientBoostingClassifier(
    max_iter=200, learning_rate=0.1, max_depth=5, random_state=42
)
hgb.fit(X_tr_l, y_tr_l)
time_hist = time.time() - start

print(f"GradientBoosting:     {time_standard:.2f}s  "
      f"accuracy={gb_standard.score(X_te_l, y_te_l):.4f}")
print(f"HistGradientBoosting: {time_hist:.2f}s  "
      f"accuracy={hgb.score(X_te_l, y_te_l):.4f}")
print(f"\nSpeedup: {time_standard / time_hist:.1f}x")

In [ ]:
# HistGradientBoosting with early stopping
hgb_early = HistGradientBoostingClassifier(
    max_iter=1000,
    learning_rate=0.1,
    max_depth=5,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42
)
hgb_early.fit(X_tr_l, y_tr_l)

print(f"Max iterations set:   1000")
print(f"Iterations used:      {hgb_early.n_iter_}")
print(f"Test accuracy:        {hgb_early.score(X_te_l, y_te_l):.4f}")

Early stopping automatically halts training when the validation score stops improving,
preventing overfitting and saving computation.

---
## 10. XGBoost, LightGBM, and CatBoost

While scikit-learn's gradient boosting implementations are solid, three external
libraries dominate competitive machine learning:

| Library | Key innovation | Strengths |
|---------|----------------|----------|
| **XGBoost** | Regularised objective, parallel tree building | Fast, flexible, the original competition winner |
| **LightGBM** | Histogram-based, leaf-wise growth (GOSS + EFB) | Extremely fast on large datasets |
| **CatBoost** | Ordered boosting, native categorical handling | Best for data with many categorical features |

All three follow the same gradient boosting framework. Their APIs are very similar:

```python
# XGBoost
import xgboost as xgb
model = xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

# LightGBM
import lightgbm as lgb
model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

# CatBoost
from catboost import CatBoostClassifier
model = CatBoostClassifier(iterations=200, learning_rate=0.1, depth=5)
model.fit(X_train, y_train)
```

For most tabular problems, any of these three (or sklearn's HistGradientBoosting) will
give excellent results. The choice often comes down to specific features you need
(categorical handling, GPU support, etc.) or personal preference.

---
## 11. The Grand Comparison: Decision Tree vs Random Forest vs Gradient Boosting

Let us compare all three approaches on the same dataset using cross-validation.

In [ ]:
# Multi-class classification dataset
X_comp, y_comp = make_classification(
    n_samples=2000, n_features=20, n_informative=12,
    n_redundant=4, n_classes=4, n_clusters_per_class=2,
    random_state=42
)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_comp, y_comp, test_size=0.25, random_state=42
)

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=200, learning_rate=0.1, random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.1, max_depth=5, random_state=42
    ),
}

results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_tr_c, y_tr_c, cv=5, scoring='accuracy', n_jobs=-1)
    model.fit(X_tr_c, y_tr_c)
    test_acc = model.score(X_te_c, y_te_c)
    results[name] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_acc': test_acc,
        'cv_scores': cv_scores,
    }
    print(f"{name:25s}  CV: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}  "
          f"Test: {test_acc:.4f}")

In [ ]:
# Bar chart comparison
names = list(results.keys())
cv_means = [results[n]['cv_mean'] for n in names]
cv_stds = [results[n]['cv_std'] for n in names]
test_accs = [results[n]['test_acc'] for n in names]

x_pos = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x_pos - width/2, cv_means, width, yerr=cv_stds,
               label='CV accuracy', capsize=5, color='steelblue')
bars2 = ax.bar(x_pos + width/2, test_accs, width,
               label='Test accuracy', color='coral')

ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Model comparison: CV and test accuracy')
ax.legend()
ax.set_ylim(0.6, 1.0)

# Annotate bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation score distributions
fig, ax = plt.subplots(figsize=(10, 5))

all_cv_scores = [results[n]['cv_scores'] for n in names]
bp = ax.boxplot(all_cv_scores, patch_artist=True, labels=names)

colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#ff66b3']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('CV Accuracy')
ax.set_title('Cross-validation score distribution by model')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---
## 12. Decision Boundary Comparison

Let us visualise the decision boundaries of all three approaches on the 2D moons
dataset.

In [ ]:
models_2d = {
    'Decision Tree\n(depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest\n(200 trees)': RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting\n(200 trees, lr=0.1)': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42
    ),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models_2d.items()):
    model.fit(X_train, y_train)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdBu',
               edgecolors='k', s=30)
    ax.set_title(f'{name}\nTest: {model.score(X_test, y_test):.3f}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Decision boundaries: tree vs forest vs boosting', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

- The **single tree** has jagged, rectangular boundaries with isolated pockets.
- The **random forest** smooths out the boundaries by averaging many trees.
- **Gradient boosting** produces the smoothest boundaries, built up incrementally.

---
## 13. Summary

| Method | Strategy | Trees | Key strength | Key weakness |
|--------|----------|-------|-------------|-------------|
| **AdaBoost** | Reweight samples | Sequential, shallow | Simple, effective for binary | Sensitive to outliers and noise |
| **Gradient Boosting** | Fit residuals | Sequential, shallow | Highest accuracy on tabular data | Slow to train, many hyperparameters |
| **HistGradientBoosting** | Histogram-based GB | Sequential, shallow | Fast even on large datasets | Less flexible loss functions |
| **Random Forest** | Average independent trees | Parallel, deep | Robust, few hyperparameters | Cannot match GB accuracy on many tasks |

### Practical guidelines

1. **Start with a Random Forest** — fast, robust, and gives a strong baseline.
2. **Switch to Gradient Boosting** when you need maximum accuracy and can afford tuning.
3. **Use HistGradientBoosting** (or XGBoost/LightGBM) for datasets with more than
   ~10,000 samples.
4. **Tune learning_rate and n_estimators together** — lower rates with more trees.
5. **Use early stopping** to prevent overfitting and save compute.
6. **Keep max_depth shallow** for boosting (3-5), deep for random forests.

Tree-based methods (especially gradient boosting) remain the dominant approach for
structured/tabular data. They are fast, handle mixed feature types, require minimal
preprocessing, and consistently deliver top performance.